In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Stage 4: Exploratory Analysis — SupplyChain Manufacturing\n",
    "**Team:** The Leftside Undergrads\n",
    "**Course:** ISM 6562 — Big Data for Business Applications\n",
    "\n",
    "This notebook contains ad-hoc analysis and visualizations built on top of the curated and analytics outputs from Stages 1–3."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pyspark.sql import SparkSession\n",
    "import pyspark.sql.functions as F\n",
    "import matplotlib.pyplot as plt\n",
    "import pandas as pd\n",
    "\n",
    "spark = SparkSession.builder \\\n",
    "    .appName('SupplyChain-Exploration') \\\n",
    "    .master('spark://spark-master:7077') \\\n",
    "    .getOrCreate()\n",
    "\n",
    "BASE      = 'hdfs://namenode:9000/data/supplychain'\n",
    "ANALYTICS = {\n",
    "    'defect_analysis':    f'{BASE}/analytics/defect_analysis',\n",
    "    'inventory_risk':     f'{BASE}/analytics/inventory_risk',\n",
    "    'supplier_scorecard': f'{BASE}/analytics/supplier_scorecard',\n",
    "    'equipment':          f'{BASE}/analytics/equipment_monitoring',\n",
    "}\n",
    "print('Spark session ready.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Defect Rate by Factory and Shift"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "defects = spark.read.parquet(f\"{ANALYTICS['defect_analysis']}/defect_by_line_shift\")\n",
    "defects.show(20, truncate=False)\n",
    "\n",
    "df = defects.toPandas()\n",
    "pivot = df.pivot_table(index='factory_id', columns='shift', values='defect_rate_pct', aggfunc='mean')\n",
    "\n",
    "pivot.plot(kind='bar', figsize=(12, 5), title='Average Defect Rate by Factory and Shift')\n",
    "plt.ylabel('Defect Rate (%)')\n",
    "plt.xlabel('Factory')\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Top Defect Codes by Frequency"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "defect_codes = spark.read.parquet(f\"{ANALYTICS['defect_analysis']}/defect_code_summary\")\n",
    "defect_codes.show(20, truncate=False)\n",
    "\n",
    "df = defect_codes.toPandas().head(10)\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.bar(df['defect_code'], df['total_defects'], color='tomato')\n",
    "plt.title('Top 10 Defect Codes by Total Frequency')\n",
    "plt.xlabel('Defect Code')\n",
    "plt.ylabel('Total Defects')\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Inventory Risk — Products Below Reorder Point"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "inventory = spark.read.parquet(ANALYTICS['inventory_risk'])\n",
    "\n",
    "at_risk = inventory.filter(F.col('at_risk') == True)\n",
    "print(f'Products at risk of stockout: {at_risk.count()}')\n",
    "at_risk.orderBy(F.asc('days_to_stockout')).show(20, truncate=False)\n",
    "\n",
    "df = at_risk.toPandas().head(15)\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.barh(df['product_id'], df['days_to_stockout'], color='orange')\n",
    "plt.title('At-Risk Products — Days Until Stockout')\n",
    "plt.xlabel('Days to Stockout')\n",
    "plt.ylabel('Product ID')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Supplier Scorecard"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "scorecard = spark.read.parquet(ANALYTICS['supplier_scorecard'])\n",
    "scorecard.show(20, truncate=False)\n",
    "\n",
    "df = scorecard.toPandas()\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "axes[0].bar(df['supplier_name'], df['avg_on_time_delivery_rate'], color='steelblue')\n",
    "axes[0].axhline(0.90, color='red', linestyle='--', label='90% target')\n",
    "axes[0].set_title('On-Time Delivery Rate by Supplier')\n",
    "axes[0].set_xlabel('Supplier')\n",
    "axes[0].set_ylabel('Rate')\n",
    "axes[0].tick_params(axis='x', rotation=45)\n",
    "axes[0].legend()\n",
    "\n",
    "axes[1].bar(df['supplier_name'], df['avg_supplier_defect_rate'], color='tomato')\n",
    "axes[1].axhline(0.05, color='red', linestyle='--', label='5% threshold')\n",
    "axes[1].set_title('Defect Rate by Supplier')\n",
    "axes[1].set_xlabel('Supplier')\n",
    "axes[1].set_ylabel('Rate')\n",
    "axes[1].tick_params(axis='x', rotation=45)\n",
    "axes[1].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Supplier Defect Rate vs Production Defect Rate"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "supplier_vs_prod = spark.read.parquet(f\"{ANALYTICS['supplier_scorecard']}/supplier_vs_production_defects\")\n",
    "supplier_vs_prod.show(20, truncate=False)\n",
    "\n",
    "df = supplier_vs_prod.toPandas()\n",
    "plt.figure(figsize=(8, 5))\n",
    "plt.scatter(df['avg_supplier_defect_rate'], df['production_defect_rate_pct'], color='purple', alpha=0.7)\n",
    "plt.title('Supplier Defect Rate vs Production Defect Rate')\n",
    "plt.xlabel('Avg Supplier Defect Rate')\n",
    "plt.ylabel('Production Defect Rate (%)')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Summary of Key Findings"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('=== KEY FINDINGS ===')\n",
    "print(f\"Total at-risk products: {at_risk.count()}\")\n",
    "print(f\"High-risk suppliers: {df[df['overall_risk'] == 'HIGH'].shape[0]}\")\n",
    "print('Exploration complete.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}